### MNIST ANN

In [16]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout,Dense,Input,BatchNormalization,Flatten
from sklearn.model_selection import train_test_split

In [17]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [18]:
x_train.shape

(60000, 28, 28)

In [26]:
model=Sequential()
model.add(Input(shape=(28,28)))
model.add(Flatten())
model.add(Dense(128,activation="relu"))
model.add(Dropout(0.1))
model.add(Dense(64,activation="relu"))
model.add(Dense(10,activation="softmax"))

In [27]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer='adam',
    metrics=['accuracy']
)

In [28]:
history = model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    batch_size=128,
    epochs=50,
    
)

Epoch 1/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7810 - loss: 4.0606 - val_accuracy: 0.9002 - val_loss: 0.5818
Epoch 2/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8722 - loss: 0.6658 - val_accuracy: 0.9277 - val_loss: 0.3200
Epoch 3/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8997 - loss: 0.4184 - val_accuracy: 0.9365 - val_loss: 0.2542
Epoch 4/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9202 - loss: 0.3049 - val_accuracy: 0.9447 - val_loss: 0.2239
Epoch 5/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9301 - loss: 0.2569 - val_accuracy: 0.9488 - val_loss: 0.2127
Epoch 6/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9404 - loss: 0.2120 - val_accuracy: 0.9503 - val_loss: 0.1810
Epoch 7/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9468 - loss: 0.1864 - val_accuracy: 0.9567 - val_loss: 0.1884
Epoch 8/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9498 - loss: 0.1723 - val_accuracy: 0.

In [30]:
test_loss,accuracy=model.evaluate(
    x_train,y_train
)
print("accuracy:",accuracy)

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9932 - loss: 0.0261
accuracy: 0.9931666851043701


In [66]:
import pandas as pd

In [67]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [68]:
def prepare_data(df: pd.DataFrame):
    df = df.copy()

    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
    
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
    
    X = df.drop(columns=['customerID', 'Churn'])
    y = df['Churn']
    
    return X, y

In [69]:
def build_preprocessor(X: pd.DataFrame):
    num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = X.select_dtypes(include=['str']).columns.tolist()  

    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    return ColumnTransformer(transformers=[
        ('nums', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ])

In [70]:
url="../data/Telco-Customer-Churn.csv"
df= pd.read_csv(url)
RANDOM_STATE=66
X,y=prepare_data(df)

In [71]:
Xtrain,Xtest,ytrain,ytest =train_test_split(X,y,test_size=0.2,stratify=y)

In [72]:
pre=build_preprocessor(Xtrain)
Xtrain=pre.fit_transform(Xtrain)
Xtest=pre.transform(Xtest)


In [73]:
Xtrain.shape

(5634, 45)

In [82]:
model = Sequential(
    [
        Dense(
            64,
            activation='relu',
            input_shape=(Xtrain.shape[1],),
        ),
        Dropout(0.2),  
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid'),  
    ]
)

/home/sayak/Documents/RetailMind AI/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [83]:
model.compile(
    optimizer='adam', loss='binary_crossentropy', metrics=['accuracy',"precision"]
)

In [97]:
history = model.fit(
    Xtrain,
    ytrain,
    validation_split=0.2,
    batch_size=32,
    epochs=30
)

Epoch 1/30
  1/141 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.9688 - loss: 0.2149 - precision: 1.0000

141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8482 - loss: 0.3115 - precision: 0.7564 - val_accuracy: 0.7826 - val_loss: 0.4839 - val_precision: 0.5871
Epoch 2/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8471 - loss: 0.3209 - precision: 0.7532 - val_accuracy: 0.7782 - val_loss: 0.4845 - val_precision: 0.5745
Epoch 3/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8538 - loss: 0.3169 - precision: 0.7623 - val_accuracy: 0.7844 - val_loss: 0.4843 - val_precision: 0.5863
Epoch 4/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8485 - loss: 0.3113 - precision: 0.7532 - val_accuracy: 0.7826 - val_loss: 0.4849 - val_precision: 0.5827
Epoch 5/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8544 - loss: 0.3106 - precision: 0.7600 - val_accuracy: 0.7862 - val_loss: 0.4850 - val_precision: 0.5926
Epoch 6/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8509 - loss: 0.3145 - precision: 0.7608 - val_accuracy: 0.7853 - val_loss: 0.4855 - v

In [96]:
test_loss, test_accuracy,precision = model.evaluate(Xtest, ytest)
print(f'\nTest Accuracy: {test_accuracy:.4f}')
print(f'\nTest Precision: {precision:.4f}')

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7899 - loss: 0.5017 - precision: 0.6196

Test Accuracy: 0.7899

Test Precision: 0.6196
